In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from rembg import remove
from PIL import Image
import easygui as eg
import cv2
from fastapi import FastAPI
import os
from matplotlib.image import imread
import matplotlib.pyplot as plt
import numpy as np
from numpy import asarray
import cv2 
from PIL import Image
import pandas as pd 
import pickle
import joblib
from sklearn.ensemble import RandomForestClassifier


#%matplotlib inline

app = FastAPI()


def imageProcess():

    with open('drive/MyDrive/Colab Notebooks/RFA.pkl', 'rb') as file:
       clf=pickle.load(file)

    counter=1 
    names = ['area','perimeter','physiological_length','physiological_width','rectangularity','circularity']     
    df = pd.DataFrame(columns=names)
    input_path  = 'rfa_test.jpg'
    input1 = cv2.imread(input_path)
    resized_image = cv2.resize(input1, (1600, 1200))
    resized_path = 'resized.jpg'
    cv2.imwrite(resized_path, resized_image)
   
    output_path = 'output.jpg'
    input = cv2.imread(resized_path)
    output = remove(input)
    cv2.imwrite(output_path, output)

    #image acquisition
    input_image = imread(output_path)
            
    #RGB to gray
    grayscale_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2GRAY)
    
    #median filtering
    medianfilter_image=cv2.medianBlur(grayscale_image,3)    #3 is the kernel size

    #OTSU thresholding
    _ , binary_image=cv2.threshold(medianfilter_image,0,255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    #Inverted Image
    inverted_image = cv2.bitwise_not(binary_image)
    
    #Canny Edge Detection
    edges = cv2.Canny(medianfilter_image,100,200)

    contours, _ = cv2.findContours(inverted_image,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    for i, contour in enumerate(contours):
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour,True)

        x,y,w,h=cv2.boundingRect(contour)
        length=h
        width=w

        rectangularity=w/h
        if(area != 0):
            rectangularity = w*h/area
            print(rectangularity)
        else:
            rectangularity = 0

        if(perimeter!= 0):
            circularity=(4*np.pi*area)/(perimeter**2)
            print(circularity)
        else:
            circularity=0
        

    
    #displaying output
    # fig=plt.figure(1)
    # img1,img2=fig.add_subplot(121),fig.add_subplot(122)    
    # img1.imshow(input_image,cmap=plt.cm.get_cmap('gray'))      
    # img2.imshow(binary_image,cmap=plt.cm.get_cmap('gray'))
    # plt.show()

    #steps to create features data frame
    vector = [area,perimeter,w,h,rectangularity,circularity]
    df_temp = pd.DataFrame([vector],columns=names)
    df=df.append(df_temp,ignore_index=True)
    
    #a = pd.DataFrame()
    df.to_csv("features_x.csv")
    predicted_label=clf.predict("features_x.csv")
    print("hi")
    plants=['rasna','lemon']

    predicted_plant=plants[np.argmax(predicted_label)]

    print("Predicted plant label is: ", predicted_plant)


@app.get("/")
def root():
    #a = pd.DataFrame()
    #a.to_csv("features1.csv")
    imageProcess()
    
    

Accuracy: 0.6365853658536585
